## Preprocessing

### 01.Loading Data and Defining Feature Groups
Explanation: Deep learning models and tree models treat data differently. Tree models natively handle binary flags (0 or 1), but standardizing them (e.g., turning 0 into -0.45) ruins their interpretability. Cyclical features (like month_sin) are already perfectly bounded between -1 and 1. We must isolate the continuous features that actually need scaling (like returns and VIX) from those that don't.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import pickle

print("Block 1: Loading Data and Grouping Features...")

# 1. Load the Engineered Data
df = pd.read_csv("E:/fourth_sem/nifty_ml_hybrid/datasets/processed/nifty_engineered_features.csv")
df['date'] = pd.to_datetime(df['date'])

# --- THE FIX: Auto-generate missing calendar features if they aren't in the CSV ---
if 'is_monday' not in df.columns:
    df['is_monday'] = (df['date'].dt.dayofweek == 0).astype(int)
    print("  -> Auto-generated missing 'is_monday' feature.")
if 'is_friday' not in df.columns:
    df['is_friday'] = (df['date'].dt.dayofweek == 4).astype(int)
    print("  -> Auto-generated missing 'is_friday' feature.")
# ---------------------------------------------------------------------------------

# 2. Define Feature Groups
TARGET_COLS = ['target_ret_1d', 'target_dir_1d', 'target_quintile_1d']
NON_FEATURE_COLS = ['date'] + TARGET_COLS

# Identify categorical/cyclic features that should NOT be standardized
CATEGORICAL_COLS = [
    'regime_200MA', 'vol_quartile_63d', 'is_monday', 'is_friday',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos'
]

# Ensure we only try to process columns that ACTUALLY exist in the dataframe
# (This prevents future KeyErrors)
CATEGORICAL_COLS = [col for col in CATEGORICAL_COLS if col in df.columns]

# All other numerical features will be scaled
NUMERICAL_COLS = [col for col in df.columns if col not in NON_FEATURE_COLS + CATEGORICAL_COLS]
FEATURE_COLS = NUMERICAL_COLS + CATEGORICAL_COLS # The final ordered feature list

print(f"Numerical features to scale: {len(NUMERICAL_COLS)}")
print(f"Categorical/Cyclic features to bypass scaling: {len(CATEGORICAL_COLS)}")

Block 1: Loading Data and Grouping Features...
  -> Auto-generated missing 'is_monday' feature.
  -> Auto-generated missing 'is_friday' feature.
Numerical features to scale: 39
Categorical/Cyclic features to bypass scaling: 8


### 02.Chronological Train/Val/Test Split
Explanation: In standard machine learning (like predicting house prices), you shuffle data before splitting. In quantitative finance, shuffling is a fatal error. If you shuffle, you might use data from 2023 to train a model to predict a day in 2021. We must strictly slice the data chronologically: 70% for training, 15% for validation (tuning hyperparameters), and 15% for out-of-sample testing.

In [2]:
print("\nBlock 2: Chronological Splitting...")

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print(f"Train set: {train_df['date'].min().date()} to {train_df['date'].max().date()} ({len(train_df)} days)")
print(f"Val set  : {val_df['date'].min().date()} to {val_df['date'].max().date()} ({len(val_df)} days)")
print(f"Test set : {test_df['date'].min().date()} to {test_df['date'].max().date()} ({len(test_df)} days)")


Block 2: Chronological Splitting...
Train set: 2015-01-02 to 2022-08-23 (1879 days)
Val set  : 2022-08-24 to 2024-04-12 (403 days)
Test set : 2024-04-15 to 2025-11-27 (403 days)


### 03.Strict Leakage Free Scaling
Explanation: If you apply StandardScaler to the entire dataset before splitting, the mean and variance of the 2024 test data will "leak" into the 2018 training data. You must fit the scaler strictly on the train_df, and then simply .transform() the validation and test sets using the training parameters.

In [3]:
print("\nBlock 3: Leakage-Free Scaling...")

scaler = StandardScaler()

# Fit ONLY on the training numericals, then transform
train_df[NUMERICAL_COLS] = scaler.fit_transform(train_df[NUMERICAL_COLS])

# Transform val and test using the train-fitted scaler
val_df[NUMERICAL_COLS] = scaler.transform(val_df[NUMERICAL_COLS])
test_df[NUMERICAL_COLS] = scaler.transform(test_df[NUMERICAL_COLS])

# Save the scaler. In production deployment, you will load this to scale tomorrow's live data!
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
    
print("Scaling complete and scaler.pkl saved.")


Block 3: Leakage-Free Scaling...
Scaling complete and scaler.pkl saved.


### 04.Extract Tabular Arrays(For XGBOOST)
Explanation: Tree models operate on 2D matrices where each row is an independent observation (Samples, Features). We extract the raw numpy arrays for the tabular tower of your Meta-Learner.

In [4]:
print("\nBlock 4: Extracting 2D Arrays for XGBoost...")

X_train_2d = train_df[FEATURE_COLS].values
y_train_reg = train_df['target_ret_1d'].values
y_train_cls = train_df['target_dir_1d'].values

X_val_2d = val_df[FEATURE_COLS].values
y_val_reg = val_df['target_ret_1d'].values
y_val_cls = val_df['target_dir_1d'].values

X_test_2d = test_df[FEATURE_COLS].values
y_test_reg = test_df['target_ret_1d'].values
y_test_cls = test_df['target_dir_1d'].values

print(f"X_train_2d shape: {X_train_2d.shape}")


Block 4: Extracting 2D Arrays for XGBoost...
X_train_2d shape: (1879, 47)


### 05.Generate Overleafing 3D Sequences(For TFT/N-BEATS)
Explanation: The Temporal Fusion Transformer needs to "look back" in time. We use a sliding window of 30 days.
Sequence 1: Days 1 to 30 $\rightarrow$ Predicts target of Day 30 (which, remember, was shifted in feature engineering to be the return of Day 31).
Sequence 2: Days 2 to 31 $\rightarrow$ Predicts target of Day 31.
Because the first sequence requires 30 days of history to generate, the total number of target rows will drop by exactly lookback (30).

In [5]:
print("\nBlock 5: Generating 3D Sequences for Deep Learning...")

def create_sequences(X, y_reg, y_cls, lookback=30):
    Xs, ys_reg, ys_cls = [], [], []
    for i in range(len(X) - lookback):
        # Slice a window of 'lookback' days
        Xs.append(X[i:(i + lookback)])
        # The target aligns with the LAST day of the window
        ys_reg.append(y_reg[i + lookback - 1]) 
        ys_cls.append(y_cls[i + lookback - 1])
    return np.array(Xs), np.array(ys_reg), np.array(ys_cls)

LOOKBACK = 30

X_train_3d, y_train_reg_3d, y_train_cls_3d = create_sequences(X_train_2d, y_train_reg, y_train_cls, LOOKBACK)
X_val_3d, y_val_reg_3d, y_val_cls_3d = create_sequences(X_val_2d, y_val_reg, y_val_cls, LOOKBACK)
X_test_3d, y_test_reg_3d, y_test_cls_3d = create_sequences(X_test_2d, y_test_reg, y_test_cls, LOOKBACK)

print(f"X_train_3d shape: {X_train_3d.shape} (Batches, Lookback, Features)")
print(f"y_train_3d shape: {y_train_reg_3d.shape}")


Block 5: Generating 3D Sequences for Deep Learning...
X_train_3d shape: (1849, 30, 47) (Batches, Lookback, Features)
y_train_3d shape: (1849,)


### 06.Saving the Processed Artifacts
Explanation: Compressing all these arrays into a single .npz file keeps your project workspace clean. When you create your Jupyter Notebooks for model training (05_XGBoost.ipynb and 06_TFT.ipynb), you simply load this one file.


In [6]:
print("\nBlock 6: Saving Artifacts...")

np.savez("preprocessed_data.npz", 
         # 2D Tabular Data (XGBoost)
         X_train_2d=X_train_2d, X_val_2d=X_val_2d, X_test_2d=X_test_2d,
         
         # 3D Sequence Data (Transformer/LSTM)
         X_train_3d=X_train_3d, X_val_3d=X_val_3d, X_test_3d=X_test_3d,
         
         # Targets (Aligns with 2D)
         y_train_reg=y_train_reg, y_val_reg=y_val_reg, y_test_reg=y_test_reg,
         y_train_cls=y_train_cls, y_val_cls=y_val_cls, y_test_cls=y_test_cls,
         
         # Targets (Aligns with 3D)
         y_train_reg_3d=y_train_reg_3d, y_val_reg_3d=y_val_reg_3d, y_test_reg_3d=y_test_reg_3d,
         y_train_cls_3d=y_train_cls_3d, y_val_cls_3d=y_val_cls_3d, y_test_cls_3d=y_test_cls_3d
        )

print("All preprocessing arrays successfully saved to 'preprocessed_data.npz'.")


Block 6: Saving Artifacts...
All preprocessing arrays successfully saved to 'preprocessed_data.npz'.
